# AlphaZero TicTacToe — Post-Training Evaluation

Loads a trained checkpoint and runs comprehensive diagnostics:
- Win/draw/loss rates against multiple opponents
- Color symmetry check (agent should play equally well as X and O)
- Policy heatmaps on canonical board positions
- Value head calibration on known positions
- Game length distribution
- Opening book analysis (3×3)

## Pass/fail criteria

| Metric | Target |
|---|---|
| Win rate vs Random | ≥ 95% |
| Loss rate vs MinimaxAB(depth=4) | ≤ 10% |
| Self-play draw rate | ≥ 70% |
| Policy entropy | Decreasing vs untrained baseline |

In [ ]:
# ── Cell 1: GPU check ──────────────────────────────────────────────────────
import torch

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if device.type == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')

In [ ]:
# ── Cell 2: Mount Drive + Clone repo ─────────────────────────────────────
import os, sys, subprocess
from google.colab import drive

drive.mount('/content/drive')

REPO_URL = 'https://github.com/YOUR_USERNAME/TicTacToe.git'  # <-- edit this

if not os.path.exists('/content/TicTacToe'):
    subprocess.run(['git', 'clone', REPO_URL, '/content/TicTacToe'], check=True)
else:
    print('Repository already cloned.')

sys.path.insert(0, '/content/TicTacToe')
print('Ready.')

In [ ]:
# ── Cell 3: Evaluation configuration ─────────────────────────────────────
# Path to a saved checkpoint WITHOUT the .pt extension.
# Example: '/content/drive/MyDrive/alphazero_checkpoints/checkpoint_iter_0150'
CHECKPOINT_PATH = '/content/drive/MyDrive/alphazero_checkpoints/checkpoint_iter_0150'

N = 3                     # Board dimension (must match the trained checkpoint)
K = 3                     # Win length (must match the trained checkpoint)
NETWORK_TYPE = 'float32'  # Must match the network_type used during training

# Use more simulations at eval time — no training overhead
NUM_SIMULATIONS = 200

# Games per evaluation matchup (split evenly as X and O)
EVAL_GAMES = 100

print(f'Checkpoint: {CHECKPOINT_PATH}')
print(f'Board: {N}x{N}, k={K}, network={NETWORK_TYPE}, sims={NUM_SIMULATIONS}')

In [ ]:
# ── Cell 4: Imports ───────────────────────────────────────────────────────
import json
import math
import random

import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import numpy as np
import torch

from tictactoe.agents.classic_search.minimax_ab import MinimaxAB
from tictactoe.agents.random_agent import RandomAgent
from tictactoe.agents.reinforcement_learning.alphazero import AlphaZeroAgent
from tictactoe.agents.reinforcement_learning.shared.neural_net import (
    DEVICE, encode_board_flat,
)
from tictactoe.agents.reinforcement_learning.shared.self_play_trainer import SelfPlayTrainer
from tictactoe.core.board import Board
from tictactoe.core.state import GameState
from tictactoe.core.types import Cell, Player, Result

print('All imports OK.')

In [ ]:
# ── Cell 5: Load trained agent ────────────────────────────────────────────
agent = AlphaZeroAgent(
    n=N, k=K,
    num_simulations=NUM_SIMULATIONS,
    network_type=NETWORK_TYPE,
)
agent.load(CHECKPOINT_PATH)

n_params = sum(p.numel() for p in agent._net.parameters())
print(f'Agent: {agent.get_name()}')
print(f'Network parameters: {n_params:,}')
print(f'Network device: {next(agent._net.parameters()).device}')

# Also load metadata if available
meta_path = CHECKPOINT_PATH + '_meta.json'
if os.path.exists(meta_path):
    with open(meta_path) as f:
        meta = json.load(f)
    print(f'Trained for {meta["iteration"]} iterations (saved {meta.get("timestamp", "?")})')
    history = meta.get('history', {})
else:
    print('No metadata file found.')
    history = {}

In [ ]:
# ── Cell 6: Evaluation helper ─────────────────────────────────────────────

def evaluate_vs(agent, opponent, n_games: int, n: int, k: int) -> dict:
    """Evaluate agent against opponent, playing n_games/2 as X and n_games/2 as O."""
    wins = draws = losses = 0
    trainer = SelfPlayTrainer(n=n, k=k)
    half = n_games // 2

    for _ in range(half):
        _, result = trainer.play_episode(agent, opponent)
        if result == Result.DRAW: draws += 1
        elif result == Result.X_WINS: wins += 1
        else: losses += 1

    for _ in range(half):
        _, result = trainer.play_episode(opponent, agent)
        if result == Result.DRAW: draws += 1
        elif result == Result.O_WINS: wins += 1
        else: losses += 1

    total = wins + draws + losses
    return {'win_rate': wins/total, 'draw_rate': draws/total, 'loss_rate': losses/total,
            'wins': wins, 'draws': draws, 'losses': losses, 'total': total}


def evaluate_by_color(agent, opponent, n_games_each: int, n: int, k: int) -> dict:
    """Separate win rates as X and as O to detect positional bias."""
    trainer = SelfPlayTrainer(n=n, k=k)

    wx = wd_x = wl_x = 0
    for _ in range(n_games_each):
        _, result = trainer.play_episode(agent, opponent)
        if result == Result.X_WINS: wx += 1
        elif result == Result.DRAW: wd_x += 1
        else: wl_x += 1

    wo = wd_o = wl_o = 0
    for _ in range(n_games_each):
        _, result = trainer.play_episode(opponent, agent)
        if result == Result.O_WINS: wo += 1
        elif result == Result.DRAW: wd_o += 1
        else: wl_o += 1

    return {
        'wr_as_x': wx/n_games_each, 'dr_as_x': wd_x/n_games_each, 'lr_as_x': wl_x/n_games_each,
        'wr_as_o': wo/n_games_each, 'dr_as_o': wd_o/n_games_each, 'lr_as_o': wl_o/n_games_each,
    }

print('Helpers defined.')

In [ ]:
# ── Cell 7: Win/draw/loss vs four opponents ───────────────────────────────
# Four opponents in increasing difficulty:
#   1. RandomAgent       — random legal moves
#   2. MinimaxAB(d=2)    — weak search (2 ply)
#   3. MinimaxAB(d=4)    — near-perfect oracle at n=3
#   4. Untrained AZ      — random-weight AlphaZero baseline

untrained_baseline = AlphaZeroAgent(n=N, k=K, num_simulations=50, network_type=NETWORK_TYPE)

opponents = [
    ('Random',         RandomAgent()),
    ('MinimaxAB-d2',   MinimaxAB(depth=2)),
    ('MinimaxAB-d4',   MinimaxAB(depth=4)),
    ('AZ-Untrained',   untrained_baseline),
]

results_table = []
print(f'{"Opponent":22s} | {"Win%":>6} | {"Draw%":>6} | {"Loss%":>6} | {"Games":>5}')
print('-' * 58)

for name, opp in opponents:
    r = evaluate_vs(agent, opp, EVAL_GAMES, N, K)
    results_table.append((name, r))
    print(f'{name:22s} | {r["win_rate"]:>6.1%} | {r["draw_rate"]:>6.1%} | {r["loss_rate"]:>6.1%} | {r["total"]:>5}')

In [ ]:
# ── Cell 8: Color symmetry check ─────────────────────────────────────────
# A well-trained agent should play equally well as X and as O.
# Large asymmetry indicates the network has overfit to one color.

print('Win rates by color (playing against RandomAgent):')
print(f'{"":22s} | {"As X":>12} | {"As O":>12} | {"Symmetry":>9}')
print('-' * 62)

for name, opp in opponents[:3]:  # Skip untrained baseline
    r = evaluate_by_color(agent, opp, EVAL_GAMES // 2, N, K)
    symmetry = 1.0 - abs(r['wr_as_x'] - r['wr_as_o'])
    x_str = f"W={r['wr_as_x']:.1%} D={r['dr_as_x']:.1%}"
    o_str = f"W={r['wr_as_o']:.1%} D={r['dr_as_o']:.1%}"
    sym_str = f"{symmetry:.3f}"
    flag = '' if symmetry >= 0.90 else ' ← asymmetric'
    print(f'{name:22s} | {x_str:>12} | {o_str:>12} | {sym_str:>9}{flag}')

print()
print('Symmetry = 1 - |WR_as_X - WR_as_O|. Target: ≥ 0.90')

In [ ]:
# ── Cell 9: Policy heatmaps ───────────────────────────────────────────────
# Visualise the network's policy probability on four canonical positions.
# Well-trained 3x3 agent should favour the center on an empty board.

def build_board(moves_x, moves_o, n):
    """Build a Board2D from lists of X and O move coordinates."""
    board = Board.create(n)
    for r, c in moves_x:
        board[r][c] = Cell.X
    for r, c in moves_o:
        board[r][c] = Cell.O
    return board


def plot_policy_heatmap(ax, agent, board, player: Player, n: int, title: str):
    """Plot policy probability heatmap with piece symbols overlaid."""
    x = encode_board_flat(board, player, n)
    with torch.no_grad():
        logits, value = agent._net.forward(x)
        policy = torch.softmax(logits, dim=-1).cpu().reshape(n, n).numpy()

    im = ax.imshow(policy, cmap='YlOrRd', vmin=0, vmax=policy.max())
    for r in range(n):
        for c in range(n):
            cell = board[r][c]
            prob_str = f'{policy[r, c]:.2f}'
            color = 'white' if policy[r, c] > policy.max() * 0.6 else 'black'
            if cell == Cell.X:
                ax.text(c, r, 'X', ha='center', va='center', fontsize=16,
                        color='blue', fontweight='bold')
            elif cell == Cell.O:
                ax.text(c, r, 'O', ha='center', va='center', fontsize=16,
                        color='red', fontweight='bold')
            else:
                ax.text(c, r, prob_str, ha='center', va='center',
                        fontsize=9, color=color)

    ax.set_title(f'{title}\nValue={float(value):.3f} ({player.name} to move)', fontsize=9)
    ax.set_xticks(range(n)); ax.set_yticks(range(n))
    ax.set_xticklabels(range(n)); ax.set_yticklabels(range(n))
    return im


# Four canonical positions
positions = [
    ('Empty board (X to move)', Board.create(N), Player.X),
    ('After X:center (O to move)',
     build_board([(N//2, N//2)], [], N), Player.O),
    ('Midgame: X has top row, O has center',
     build_board([(0,0),(0,1)], [(1,1)], N), Player.X),
    ('X can win: needs (0,2) to complete top row',
     build_board([(0,0),(0,1),(1,1)], [(1,0),(2,2)], N), Player.X),
]

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
fig.suptitle(f'Policy Heatmaps — Trained AlphaZero (n={N}, k={K})', fontsize=12, fontweight='bold')

for ax, (title, board, player) in zip(axes, positions):
    plot_policy_heatmap(ax, agent, board, player, N, title)

plt.tight_layout()
plt.show()
print('Heatmaps: numbers show P(move) from network. X/O show occupied cells.')
print('Well-trained 3x3 agent should heavily favour center (1,1) on an empty board.')

In [ ]:
# ── Cell 10: Value head calibration ──────────────────────────────────────
# Test the value head on positions with known expected values:
#   Empty board        → ~0.0 (perfectly even)
#   X-to-win-next      → ~+1.0 (current player is about to win)
#   O-is-about-to-win  → ~-1.0 (current player is about to lose)

def get_value(agent, board, player: Player, n: int) -> float:
    x = encode_board_flat(board, player, n)
    with torch.no_grad():
        _, value = agent._net.forward(x)
    return float(value)


test_positions = [
    ('Empty 3x3 board (X)',       Board.create(N), Player.X,   0.0,  0.4),
    ('Empty 3x3 board (O)',       Board.create(N), Player.O,   0.0,  0.4),
    ('X has 2-in-a-row (X)',
     build_board([(0,0),(0,1)], [(1,1)], N), Player.X,         0.5,  1.0),
    ('X has 2-in-a-row (O)',
     build_board([(0,0),(0,1)], [(1,1)], N), Player.O,        -1.0, -0.3),
    ('X can win immediately (X)',
     build_board([(0,0),(0,1)], [(1,1),(2,2)], N), Player.X,   0.8,  1.0),
    ('O can win immediately (X)',
     build_board([(0,0),(2,2)], [(1,0),(1,1)], N), Player.X,  -1.0, -0.5),
]

print(f'{"Position":40s} | {"Predicted":>10} | {"Expected":>13} | {"Pass?":>6}')
print('-' * 78)

passes = 0
for label, board, player, lo, hi in test_positions:
    val = get_value(agent, board, player, N)
    ok = lo <= val <= hi
    passes += int(ok)
    print(f'{label:40s} | {val:>10.4f} | {f"[{lo:.1f}, {hi:.1f}]":>13} | {"PASS" if ok else "FAIL":>6}')

print(f'\nPassed {passes}/{len(test_positions)} value calibration checks.')

In [ ]:
# ── Cell 11: Game length distribution ────────────────────────────────────
# Distribution of self-play game lengths.
# For 3x3: optimal play always ends in exactly 5-9 moves.
# Short games (5-6 moves) with many wins indicate one player dominates.
# Many 9-move draws indicate balanced, near-optimal play.

trainer = SelfPlayTrainer(n=N, k=K)
game_results = []
game_lens = []
N_GAMES = 200

for _ in range(N_GAMES):
    moves, result = trainer.play_episode(agent, agent)
    game_lens.append(len(moves))
    game_results.append(result)

draws = sum(1 for r in game_results if r == Result.DRAW)
x_wins = sum(1 for r in game_results if r == Result.X_WINS)
o_wins = sum(1 for r in game_results if r == Result.O_WINS)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
fig.suptitle(f'Self-Play Statistics — {N_GAMES} games (n={N}, k={K})', fontsize=12)

# Game length histogram
ax = axes[0]
all_lengths = range(min(game_lens), max(game_lens) + 2)
ax.hist(game_lens, bins=all_lengths, edgecolor='black', alpha=0.7, color='steelblue')
mean_len = sum(game_lens) / len(game_lens)
ax.axvline(mean_len, color='red', linestyle='--', label=f'Mean={mean_len:.1f}')
ax.set_title('Game Length Distribution')
ax.set_xlabel('Number of moves'); ax.set_ylabel('Frequency')
ax.legend(); ax.grid(True, alpha=0.4)

# Result pie chart
ax = axes[1]
labels = [f'X Wins ({x_wins})', f'Draws ({draws})', f'O Wins ({o_wins})']
sizes = [x_wins, draws, o_wins]
colors = ['lightblue', 'lightgreen', 'lightsalmon']
ax.pie([s if s > 0 else 0.001 for s in sizes], labels=labels, colors=colors,
       autopct='%1.1f%%', startangle=90)
ax.set_title('Game Outcome Distribution')

plt.tight_layout()
plt.show()

print(f'Self-play draw rate: {draws/N_GAMES:.1%} (target ≥ 70% for well-trained 3x3 agent)')
print(f'Mean game length:    {mean_len:.1f} moves')

In [ ]:
# ── Cell 12: Opening book analysis (3×3 only) ─────────────────────────────
# Show the network's probability distribution over all 9 moves on an empty
# 3x3 board. A well-trained agent should heavily favour the center (1,1).

if N == 3:
    empty_board = Board.create(3)
    x = encode_board_flat(empty_board, Player.X, 3)
    with torch.no_grad():
        logits, value = agent._net.forward(x)
        probs = torch.softmax(logits, dim=-1).cpu().reshape(3, 3).numpy()

    labels = [['TL', 'TC', 'TR'], ['ML', 'MC', 'MR'], ['BL', 'BC', 'BR']]

    fig, axes = plt.subplots(1, 2, figsize=(10, 4))
    fig.suptitle('Opening Book Analysis — Empty 3x3 Board (X to move)', fontsize=12)

    # Heatmap
    ax = axes[0]
    im = ax.imshow(probs, cmap='YlOrRd', vmin=0)
    for r in range(3):
        for c in range(3):
            color = 'white' if probs[r, c] > probs.max() * 0.6 else 'black'
            ax.text(c, r, f'{labels[r][c]}\n{probs[r,c]:.3f}',
                    ha='center', va='center', fontsize=10, color=color)
    ax.set_title('Policy Probabilities')
    ax.set_xticks([]); ax.set_yticks([])
    plt.colorbar(im, ax=ax)

    # Bar chart
    ax = axes[1]
    position_names = [labels[r][c] for r in range(3) for c in range(3)]
    position_probs = [probs[r, c] for r in range(3) for c in range(3)]
    bar_colors = ['gold' if name == 'MC' else
                  'steelblue' if name in ('TL','TR','BL','BR') else
                  'lightcoral' for name in position_names]
    ax.bar(position_names, position_probs, color=bar_colors, edgecolor='black')
    ax.set_title('Probability by Position')
    ax.set_xlabel('Position (MC=Center, corners=blue, edges=red)')
    ax.set_ylabel('P(move)')
    ax.grid(True, alpha=0.4, axis='y')

    plt.tight_layout()
    plt.show()

    center_p = probs[1, 1]
    corner_p = (probs[0,0]+probs[0,2]+probs[2,0]+probs[2,2])/4
    edge_p   = (probs[0,1]+probs[1,0]+probs[1,2]+probs[2,1])/4

    print(f'Center (MC):      {center_p:.4f}')
    print(f'Corners (avg):    {corner_p:.4f}')
    print(f'Edges (avg):      {edge_p:.4f}')
    print(f'Value (from X):   {float(value):.4f}')
    print()
    if center_p > 0.25:
        print('Good: network assigns above-chance probability to center.')
    else:
        print('Weak: network does not clearly prefer center — may need more training.')
else:
    print(f'Opening book analysis only implemented for n=3 (current n={N}).')
    print('For larger boards, use the policy heatmap in Cell 9.')

In [ ]:
# ── Cell 13: Plot training history (if available) ─────────────────────────
# Reconstruct training curves from the checkpoint metadata.

import math

if history and 'iteration' in history and len(history['iteration']) > 1:
    iters = history['iteration']
    eval_iters = [i for i, v in zip(iters, history.get('vs_random_wr', []))
                  if v is not None]
    wr_random  = [v for v in history.get('vs_random_wr', []) if v is not None]
    wr_minimax = [v for v in history.get('vs_minimax_wr', []) if v is not None]

    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    fig.suptitle('Training History (from checkpoint metadata)', fontsize=12)

    axes[0].plot(iters, history['avg_loss'], 'b-', linewidth=1)
    axes[0].set_title('Training Loss'); axes[0].set_xlabel('Iteration')
    axes[0].set_ylabel('Loss'); axes[0].grid(True, alpha=0.4)

    if eval_iters:
        axes[1].plot(eval_iters, wr_random, 'g-o', markersize=4, label='vs Random')
        axes[1].plot(eval_iters, wr_minimax, 'r-o', markersize=4, label='vs Minimax')
        axes[1].axhline(0.95, color='green', linestyle='--', alpha=0.4)
        axes[1].axhline(0.50, color='red', linestyle='--', alpha=0.4)
    axes[1].set_title('Win Rates'); axes[1].set_xlabel('Iteration')
    axes[1].set_ylim(-0.05, 1.05); axes[1].legend(); axes[1].grid(True, alpha=0.4)

    if 'draw_rate_selfplay' in history:
        axes[2].plot(iters, history['draw_rate_selfplay'], color='purple', linewidth=1)
        axes[2].axhline(0.70, color='gray', linestyle='--', alpha=0.5, label='70% target')
        axes[2].set_title('Self-Play Draw Rate')
        axes[2].set_xlabel('Iteration'); axes[2].set_ylim(-0.05, 1.05)
        axes[2].legend(); axes[2].grid(True, alpha=0.4)

    plt.tight_layout()
    plt.show()
else:
    print('No training history found in checkpoint metadata.')

In [ ]:
# ── Cell 14: Summary report ───────────────────────────────────────────────
# Pass/fail assessment vs the four training success criteria.

print('=' * 60)
print(f'EVALUATION SUMMARY — {agent.get_name()}')
print('=' * 60)
print()

# Find Random and MinimaxAB-d4 results
results_dict = {name: r for name, r in results_table}

print('Win / Draw / Loss rates:')
for name, r in results_table:
    print(f'  vs {name:20s}: W={r["win_rate"]:>6.1%}  D={r["draw_rate"]:>6.1%}  L={r["loss_rate"]:>6.1%}')

print()
print('Pass/Fail against training success criteria:')

checks = []

# Criterion 1: ≥95% win rate vs Random
if 'Random' in results_dict:
    wr = results_dict['Random']['win_rate']
    ok = wr >= 0.95
    checks.append(ok)
    flag = 'PASS' if ok else 'FAIL'
    print(f'  [{flag}] Win rate vs Random ≥ 95%       → {wr:.1%}')

# Criterion 2: ≤10% loss rate vs MinimaxAB-d4
if 'MinimaxAB-d4' in results_dict:
    lr = results_dict['MinimaxAB-d4']['loss_rate']
    ok = lr <= 0.10
    checks.append(ok)
    flag = 'PASS' if ok else 'FAIL'
    print(f'  [{flag}] Loss rate vs MinimaxAB-d4 ≤ 10% → {lr:.1%}')

# Criterion 3: self-play draw rate ≥70%
sp_draw = draws / N_GAMES
ok = sp_draw >= 0.70
checks.append(ok)
flag = 'PASS' if ok else 'FAIL'
print(f'  [{flag}] Self-play draw rate ≥ 70%         → {sp_draw:.1%}')

# Criterion 4: center preference on empty board
if N == 3:
    ok = probs[1, 1] > 1/9  # above chance
    checks.append(ok)
    flag = 'PASS' if ok else 'FAIL'
    print(f'  [{flag}] Center preference (P > 1/9 = {1/9:.3f}) → {probs[1,1]:.3f}')

print()
n_pass = sum(checks)
n_total = len(checks)
print(f'Overall: {n_pass}/{n_total} criteria passed')
if n_pass == n_total:
    print('Training successful — agent plays near-optimally.')
elif n_pass >= n_total * 0.75:
    print('Mostly trained — consider 50-100 more iterations.')
else:
    print('Undertrained — agent needs significantly more self-play iterations.')